# 从传统 NLP 到大模型工程师：Colab 项目实战手册

> 目标用户：有后端工程经验 + 传统 NLP 基础（TF-IDF、word2vec、Transformer 基础）

这个 Notebook 按照“**学习路径 + 可执行项目**”设计，覆盖：
1. 经典 Transformer 微调（文本分类）
2. 向量检索（Embedding + FAISS）
3. RAG（检索增强生成）端到端实践

建议在 **Colab GPU** 环境运行（Runtime -> Change runtime type -> GPU）。

## 0. 学习路径（建议 4~6 个月）

### 阶段A：大模型基础（2~3 周）
- Transformer 深入：注意力、位置编码、预训练/微调范式
- 生成式模型：Causal LM vs Seq2Seq LM
- Tokenizer、Embedding、Logits、Sampling（temperature/top-k/top-p）

### 阶段B：工程化与微调（4~6 周）
- Hugging Face 生态：`datasets`、`transformers`、`evaluate`
- PEFT/LoRA、QLoRA 思路
- 训练监控：loss、learning rate、过拟合识别

### 阶段C：RAG 专项（4~6 周）
- 文档切分策略（chunk size / overlap）
- 向量模型与索引（FAISS / HNSW）
- 检索策略：Dense、BM25、Hybrid、Rerank
- 生成策略：prompt 模板、上下文注入、引用追踪
- 评估：命中率、忠实性（faithfulness）、答案相关性

### 阶段D：生产化（持续）
- 推理服务化（FastAPI + vLLM/TGI）
- 缓存、限流、可观测性（Tracing + Metrics）
- 安全与治理：提示词注入、越权检索、PII 脱敏

---
**本 Notebook 对应项目：**
- 项目1：AG News 分类微调
- 项目2：SQuAD 语料向量检索
- 项目3：RAG 问答系统（检索 + 生成）

In [ ]:
# 如果在 Colab 首次运行，请先安装依赖
!pip -q install transformers datasets evaluate sentence-transformers faiss-cpu accelerate scikit-learn

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

## 1) 项目1：Transformer 微调（AG News 四分类）

**目标**：掌握标准的 Hugging Face 微调流程（数据处理、tokenize、Trainer、评估）。

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset('ag_news')
raw_ds

In [ ]:
# 为了加速 Colab 演示，使用一个小子集
train_ds = raw_ds['train'].shuffle(seed=SEED).select(range(8000))
valid_ds = raw_ds['test'].shuffle(seed=SEED).select(range(2000))

train_ds, valid_ds

In [ ]:
from transformers import AutoTokenizer

model_ckpt = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess(examples):
    return tokenizer(examples['text'], truncation=True)

train_tok = train_ds.map(preprocess, batched=True)
valid_tok = valid_ds.map(preprocess, batched=True)

cols = ['input_ids', 'attention_mask', 'label']
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in cols])
valid_tok = valid_tok.remove_columns([c for c in valid_tok.column_names if c not in cols])

In [ ]:
import evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

acc_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return acc_metric.compute(predictions=preds, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=4)

args = TrainingArguments(
    output_dir='agnews-distilbert',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='no',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## 2) 项目2：向量检索（SQuAD 文档检索）

**目标**：将文本切片向量化并建立 FAISS 索引，给定问题检索 Top-k 上下文。

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

squad = load_dataset('squad', split='train[:5000]')

# 去重 context，避免重复文档
contexts = list(dict.fromkeys(squad['context']))
print('unique contexts:', len(contexts))

embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
ctx_emb = embed_model.encode(contexts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
ctx_emb = np.asarray(ctx_emb, dtype='float32')

index = faiss.IndexFlatIP(ctx_emb.shape[1])
index.add(ctx_emb)
print('index ntotal:', index.ntotal)

In [ ]:
def retrieve(query, topk=5):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    q_emb = np.asarray(q_emb, dtype='float32')
    scores, ids = index.search(q_emb, topk)
    return [(contexts[i], float(scores[0][rank])) for rank, i in enumerate(ids[0])]

query = 'What is the capital city of France?'
results = retrieve(query, topk=3)
for i, (ctx, s) in enumerate(results, 1):
    print(f'[{i}] score={s:.4f}\n{ctx[:300]}\n')

## 3) 项目3：RAG 端到端问答（检索 + 生成）

**目标**：
1. 先检索相关文档片段
2. 将片段拼接进 Prompt
3. 用生成模型输出答案

这里使用 `google/flan-t5-base`（英文问答效果稳定，适合 Colab 演示）。

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

gen_ckpt = "google/flan-t5-base"
rag_tok = AutoTokenizer.from_pretrained(gen_ckpt)
rag_model = AutoModelForSeq2SeqLM.from_pretrained(gen_ckpt)
if torch.cuda.is_available():
    rag_model = rag_model.cuda()

def build_prompt(question, retrieved_contexts):
    context_block = "\n\n".join([f"[{i+1}] {c}" for i, c in enumerate(retrieved_contexts)])
    prompt = f"""You are a QA assistant. Answer the question only using the provided context.
If the context is insufficient, say \"I don't know based on the given context.\".

Context:
{context_block}

Question: {question}
Answer:"""
    return prompt

def rag_answer(question, topk=4, max_new_tokens=64):
    retrieved = retrieve(question, topk=topk)
    retrieved_contexts = [x[0] for x in retrieved]
    prompt = build_prompt(question, retrieved_contexts)

    inputs = rag_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    outputs = rag_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    ans = rag_tok.decode(outputs[0], skip_special_tokens=True)
    return ans, retrieved

In [ ]:
question = 'Which city is the capital of France?'
answer, retrieved = rag_answer(question)

print('Q:', question)
print('A:', answer)
print('\nTop retrieved contexts:')
for i, (ctx, score) in enumerate(retrieved, 1):
    print(f'[{i}] score={score:.4f} | {ctx[:180]}...')

## 4) RAG 进阶建议（下一步必做）

1. **Chunking 优化**：按语义段落切分，而不是固定长度。
2. **Hybrid Retrieval**：将 BM25 与向量检索融合，提升召回稳健性。
3. **Reranker**：用交叉编码器对 Top-50 重排到 Top-5。
4. **评估集构建**：用你业务中的真实问题构造黄金集，统计 Recall@k / MRR / 答案正确率。
5. **防幻觉策略**：要求模型返回引用片段 ID；无依据时必须拒答。

---
### 你可以如何把它变成求职项目
- 项目名：`Production-ready RAG QA for Domain Knowledge`
- 技术栈：`FastAPI + FAISS + SentenceTransformers + FLAN-T5`
- 加分点：
  - 多路召回（dense + sparse）
  - 召回与生成全链路监控
  - 数据更新自动增量索引